<a href="https://colab.research.google.com/github/flowingice0/ControlCameraViews/blob/main/ComfyUI_Colab(ver1_1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ComfyUI Colab 部署（工作流保存到 Google Drive）
选择 L4 GPU 后运行下面唯一代码单元。Drive 的 AI_Models、ComfyUI_Input 和 ComfyUI_Output 会复用此前目录；ComfyUI 工作流和用户设置保存到 MyDrive/Comfyui_colab/ComfyUI_User。ComfyUI 官方支持用 --user-directory 指定这个持久目录，工作流会位于其 default/workflows 子目录。

保存工作流时仍在 ComfyUI 里使用 Save / Save As。新 session 再运行本单元后，工作流应继续出现在 Workflows 列表中。首次 Drive 授权可能需要 Google 确认；公网 URL 请复制后粘贴到浏览器地址栏打开。

In [ ]:
#@title 🚀 部署 ComfyUI（含 SeedVR2 插件与模型自动关联）
import os, sys, time, subprocess, pathlib, shutil, urllib.request, re, json

# 1) 检查 Colab GPU
import torch
if not torch.cuda.is_available():
    raise RuntimeError("未检测到 CUDA GPU。请在 Runtime > Change runtime type 里选择 L4 GPU。")
print("GPU:", torch.cuda.get_device_name(0))

# 2) 挂载当前登录 Google 账号的 Drive
from google.colab import drive
MOUNT = pathlib.Path("/content/drive")
MYDRIVE = MOUNT / "MyDrive"
if not MYDRIVE.exists():
    drive.mount(str(MOUNT))
if not MYDRIVE.exists():
    raise RuntimeError("Drive 尚未挂载，请确认 Colab 的 Google Drive 授权。")

# =========================================================
# 🔄 [极重要：强行刷新 Google Drive 后台同步缓存]
# =========================================================
print("🔄 正在强行刷新 Google Drive 文件流同步缓存...")
try:
    # 逐级遍历并列出目录，迫使 drivefs 驱动更新云端最新状态
    os.listdir("/content/drive/MyDrive")
    comfy_dir = "/content/drive/MyDrive/Comfyui_colab"
    if os.path.exists(comfy_dir):
        os.listdir(comfy_dir)
        notebooks_dir = os.path.join(comfy_dir, "Colab_Notebooks")
        if os.path.exists(notebooks_dir):
            os.listdir(notebooks_dir)
            print("✨ Drive 缓存刷新成功，已强制同步最新文件列表！")
except Exception as e:
    print(f"⚠️ 刷新缓存时发生非关键异常: {e}")
# =========================================================

# Drive 持久目录
DRIVE_MODELS = MYDRIVE / "Comfyui_colab" / "AI_Models"
DRIVE_USER = MYDRIVE / "Comfyui_colab" / "ComfyUI_User"
DRIVE_CUSTOM_NODES = MYDRIVE / "Comfyui_colab" / "ComfyUI_Custom_Nodes"
DRIVE_INPUT = MYDRIVE / "Comfyui_colab" / "ComfyUI_Input"
DRIVE_OUTPUT = MYDRIVE / "Comfyui_colab" / "ComfyUI_Output"
for p in (DRIVE_MODELS, DRIVE_INPUT, DRIVE_OUTPUT):
    p.mkdir(parents=True, exist_ok=True)

# 指定专属脚本存放目录 (已更新为 Colab_Notebooks)
DRIVE_NOTEBOOKS_DIR = MYDRIVE / "Comfyui_colab" / "Colab_Notebooks"

# 3) 安装 ComfyUI 到 session 本地盘；模型和用户数据保存在 Drive
ROOT = pathlib.Path("/content/ComfyUI")
MODELS = ROOT / "models"
LOCAL_USER = ROOT / "user"
if not ROOT.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/Comfy-Org/ComfyUI.git", str(ROOT)],
        check=True,
    )

# 第一次启用 Drive 持久用户目录时，尽量迁移当前 session 里已有的工作流和设置。
if not DRIVE_USER.exists():
    if LOCAL_USER.exists():
        shutil.copytree(LOCAL_USER, DRIVE_USER, dirs_exist_ok=True)
    else:
        DRIVE_USER.mkdir(parents=True, exist_ok=True)
else:
    DRIVE_USER.mkdir(parents=True, exist_ok=True)

# 将 custom_nodes 放在 Drive，Manager 安装的节点下次 session 仍可使用。
CUSTOM_NODES = ROOT / "custom_nodes"
if CUSTOM_NODES.is_symlink():
    CUSTOM_NODES.unlink()
    if not DRIVE_CUSTOM_NODES.exists():
        DRIVE_CUSTOM_NODES.mkdir(parents=True, exist_ok=True)
elif CUSTOM_NODES.exists():
    if not DRIVE_CUSTOM_NODES.exists():
        shutil.copytree(CUSTOM_NODES, DRIVE_CUSTOM_NODES, dirs_exist_ok=True)
    shutil.rmtree(CUSTOM_NODES)
else:
    DRIVE_CUSTOM_NODES.mkdir(parents=True, exist_ok=True)
CUSTOM_NODES.symlink_to(DRIVE_CUSTOM_NODES, target_is_directory=True)

# 映射模型目录
if MODELS.is_symlink():
    MODELS.unlink()
elif MODELS.exists():
    shutil.copytree(MODELS, DRIVE_MODELS, dirs_exist_ok=True)
    shutil.rmtree(MODELS)
MODELS.symlink_to(DRIVE_MODELS, target_is_directory=True)

# 映射输入与输出目录到 Drive，方便随时查看/复用数据
LOCAL_INPUT = ROOT / "input"
if LOCAL_INPUT.is_symlink():
    LOCAL_INPUT.unlink()
elif LOCAL_INPUT.exists():
    shutil.rmtree(LOCAL_INPUT)
LOCAL_INPUT.symlink_to(DRIVE_INPUT, target_is_directory=True)

LOCAL_OUTPUT = ROOT / "output"
if LOCAL_OUTPUT.is_symlink():
    LOCAL_OUTPUT.unlink()
elif LOCAL_OUTPUT.exists():
    shutil.rmtree(LOCAL_OUTPUT)
LOCAL_OUTPUT.symlink_to(DRIVE_OUTPUT, target_is_directory=True)

# =========================================================
# 🌟 [自动步骤 A] 自动执行 SeedVR2 插件安装脚本 (从 Drive 加载)
# =========================================================
print("\n🔄 [步骤 3.5] 自动检索并配置 SeedVR2 插件...")
install_notebooks = []
if DRIVE_NOTEBOOKS_DIR.exists():
    # 使用不区分大小写的匹配查找安装文件
    for f in DRIVE_NOTEBOOKS_DIR.iterdir():
        if "seedvr2_install_custom_node" in f.name.lower() and f.name.endswith(".ipynb"):
            install_notebooks.append(f)
            break
if not install_notebooks:
    # 全局模糊检索
    for root, dirs, files in os.walk(str(MYDRIVE)):
        for file in files:
            if "seedvr2_install_custom_node" in file.lower() and file.endswith(".ipynb"):
                install_notebooks.append(pathlib.Path(root) / file)
                break
        if install_notebooks: break

if install_notebooks:
    nb_install_path = install_notebooks[0]
    print(f"🔍 找到插件安装脚本: {nb_install_path}")
    try:
        with open(nb_install_path, 'r', encoding='utf-8') as f:
            nb_data = json.load(f)
        for cell in nb_data.get('cells', []):
            if cell.get('cell_type') == 'code':
                code_text = "".join(cell.get('source', []))
                if "drive.mount" in code_text:
                    continue
                exec(code_text, globals())
        print("✅ SeedVR2 插件配置/依赖更新成功！")
    except Exception as e:
        print(f"⚠️ 自动配置 SeedVR2 插件时发生非关键异常: {e}")
else:
    print("ℹ️ 未在 Drive 找到 SeedVR2_Install_Custom_Node.ipynb，跳过。")

# =========================================================
# 🌟 [自动步骤 B] 自动执行 SeedVR2 模型目录软链接脚本
# =========================================================
print("\n🔄 [步骤 3.6] 正在自动关联 SeedVR2 大模型...")
model_link_notebooks = []
if DRIVE_NOTEBOOKS_DIR.exists():
    # 使用不区分大小写的匹配查找软连接文件
    for f in DRIVE_NOTEBOOKS_DIR.iterdir():
        if "seedvr2_link_drive_models" in f.name.lower() and f.name.endswith(".ipynb"):
            model_link_notebooks.append(f)
            break
if not model_link_notebooks:
    # 全局模糊检索
    for root, dirs, files in os.walk(str(MYDRIVE)):
        for file in files:
            if "seedvr2_link_drive_models" in file.lower() and file.endswith(".ipynb"):
                model_link_notebooks.append(pathlib.Path(root) / file)
                break
        if model_link_notebooks: break

if model_link_notebooks:
    nb_link_path = model_link_notebooks[0]
    print(f"🔍 找到模型挂载脚本: {nb_link_path}")
    try:
        with open(nb_link_path, 'r', encoding='utf-8') as f:
            nb_data = json.load(f)
        for cell in nb_data.get('cells', []):
            if cell.get('cell_type') == 'code':
                code_text = "".join(cell.get('source', []))
                if "drive.mount" in code_text:
                    continue
                exec(code_text, globals())
        print("✅ SeedVR2 模型软链接映射成功！")
    except Exception as e:
        print(f"⚠️ 自动关联 SeedVR2 模型时发生非关键异常: {e}")
else:
    print("ℹ️ 未在 Drive 找到 SeedVR2_Link_Drive_Models.ipynb，跳过。")

# =========================================================

# 4) 模型下载逻辑（略，由主框架决定）
from urllib.parse import urlparse
MODEL_LIST = []

def download_model(item):
    url = item["url"].strip()
    folder = pathlib.Path(item["folder"])
    filename = item["filename"].strip()
    if urlparse(url).scheme not in ("http", "https"):
        raise ValueError(f"只支持 HTTP(S) 直链：{url}")
    if folder.is_absolute() or ".." in folder.parts:
        raise ValueError(f"模型子目录必须是相对路径：{folder}")
    if not filename or pathlib.Path(filename).name != filename:
        raise ValueError(f"无效的文件名：{filename}")

    target_dir = MODELS / folder
    target_dir.mkdir(parents=True, exist_ok=True)
    dest = target_dir / filename

    if dest.exists() and dest.stat().st_size > 0:
        print(f"⏭️ 模型文件已存在，跳过下载: {dest.name}")
        return

    print(f"📥 开始下载: {filename} -> {target_dir}")
    if shutil.which("aria2c"):
        subprocess.run([
            "aria2c", "--summary-interval=10", "-c", "-x", "16", "-s", "16", "-j", "16",
            "-d", str(target_dir), "-o", filename, url
        ], check=True)
    else:
        urllib.request.urlretrieve(url, str(dest))
    print(f"✅ 下载完成: {filename}")

for m in MODEL_LIST:
    download_model(m)

# 5) 安装核心及基础依赖
%cd {str(ROOT)}
print("\n📦 检查并安装核心 Python 运行依赖...")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

# 安装 cloudflared 提供公网隧道穿透
if not shutil.which("cloudflared"):
    print("\n🌐 正在安装 Cloudflare Tunnel 穿透工具...")
    subprocess.run("curl -L --silent -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared", shell=True, check=True)

# 6) 后台运行 ComfyUI 与 开启 Cloudflared 服务
print("\n🚀 启动 ComfyUI 并穿透公网中...")
port = 8188
cmd_comfy = [sys.executable, "main.py", "--port", str(port), "--user-directory", str(DRIVE_USER)]
comfy_process = subprocess.Popen(cmd_comfy, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

cmd_tunnel = ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"]
tunnel_process = subprocess.Popen(cmd_tunnel, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(2)
print("\n===================== 🌐 ComfyUI 公网访问地址 =====================")
print("等待 Cloudflare 分配公网地址...")

url_found = False
for _ in range(15):
    line = tunnel_process.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        print(f"🎉 请访问以下 URL 打开 ComfyUI 界面: \n\n✨ {match.group(0)} ✨\n")
        url_found = True
        break
    time.sleep(1)

if not url_found:
    print("⚠️ 无法自动捕获 URL。请在下方日志中手动寻找 https://*.trycloudflare.com 链接：")
    for _ in range(10):
        print(tunnel_process.stdout.readline().strip())